# DSN 2026: Find the Language I Am Speaking
Team Ohùn (datapsalm) · Oludare Samuel, Victoria Fagbemiro, Sumayah Adegbite

Code and latest results: https://github.com/Dhapor/dsn2026-nigerian-language-id

In [1]:
import os, json, time
os.environ["TQDM_DISABLE"] = "1"
import numpy as np
import torch
import transformers
from collections import Counter
from common import LABELS, L2I, SR, Split, weighted_f1, augment, CLASS_W
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

## Data

In [2]:
tr, va = Split("train"), Split("validation")
print(f"{'class':6} {'train h':>8} {'val h':>6} {'clips':>6} {'speakers':>9} {'weight':>7}")
for c, lab in enumerate(LABELS):
    m = tr.labels == c
    spk = {s for s, k in zip(tr.speakers, m) if k and s not in ("", "None")}
    print(f"{lab:6} {tr.lengths[m].sum()/SR/3600:8.2f} {va.lengths[va.labels==c].sum()/SR/3600:6.2f} "
          f"{m.sum():6d} {len(spk) if spk else 'n/a':>9} {CLASS_W[c]:7.3f}")
print(Counter(tr.sources))

class   train h  val h  clips  speakers  weight
yor        9.05   0.88   3160       n/a   0.043
ibo        7.96   2.08   2592         7   0.043
hau        8.31   1.42   2755         3   0.043
ego        7.32   1.05   1631        10   0.087
anw        8.12   1.83   1912        20   0.087
efi        1.42   0.15    483         3   0.087
ibb        1.76   0.28    607         2   0.087
idu        7.16   2.12   1649        11   0.087
igl        0.48   0.12    170         2   0.087
nup        0.50   0.00    176         1   0.087
tiv        4.27   0.63   1474         2   0.087
urh        9.22   1.57   3122       n/a   0.087
other     12.98   1.85   5043        51   0.087
Counter({'afrispeech_ipa': 15817, 'hypa_voices': 4722, 'omnilingual_asr': 4235})


## Augmentation

In [3]:
rng = np.random.default_rng(0)
x = tr.wav(int(np.flatnonzero(tr.labels == L2I["tiv"])[0]), 0, 4 * SR)
for k in range(5):
    y = augment(x.copy(), rng)
    print(f"copy {k}: rms {np.sqrt(np.mean(y**2)):.3f}, corr with original {np.corrcoef(x, y)[0, 1]:.2f}")

copy 0: rms 0.025, corr with original 0.98
copy 1: rms 0.082, corr with original 1.00


copy 2: rms 0.039, corr with original 0.40
copy 3: rms 0.118, corr with original 1.00
copy 4: rms 0.161, corr with original 0.99


## Sampling weights

In [4]:
counts = np.bincount(tr.labels, minlength=len(LABELS))
for alpha in (0.0, 0.5):
    p = counts ** (1 - alpha); p = p / p.sum()
    print(f"alpha={alpha}: " + "  ".join(f"{l} {100*q:.1f}%" for l, q in zip(LABELS, p)))

alpha=0.0: yor 12.8%  ibo 10.5%  hau 11.1%  ego 6.6%  anw 7.7%  efi 1.9%  ibb 2.5%  idu 6.7%  igl 0.7%  nup 0.7%  tiv 5.9%  urh 12.6%  other 20.4%
alpha=0.5: yor 10.8%  ibo 9.7%  hau 10.0%  ego 7.7%  anw 8.4%  efi 4.2%  ibb 4.7%  idu 7.8%  igl 2.5%  nup 2.5%  tiv 7.3%  urh 10.7%  other 13.6%


## Final model config (F07)

In [5]:
args = json.load(open("runs/F07/args.json"))
{k: args[k] for k in ["base", "full", "ext", "bs", "lr", "head_lr", "epochs", "steps_per_epoch", "max_crop", "speed", "aug", "alpha", "smooth"]}

{'base': 'ajesujoba/AfriHuBERT',
 'full': True,
 'ext': ['ext_efik', 'ext_cv_ibb'],
 'bs': 16,
 'lr': 5e-05,
 'head_lr': 0.001,
 'epochs': 3.0,
 'steps_per_epoch': 1500,
 'max_crop': 8.0,
 'speed': 0.6,
 'aug': 0.8,
 'alpha': 0.5,
 'smooth': 0.1}

## Validation: recipes and model soup

In [6]:
for name, path in [("A01 debiased", "runs/soup_eval/a01.npz"), ("A03 plain", "runs/soup_eval/a03.npz"), ("soup(A01, A03)", "runs/soup_eval/soup_a01_a03.npz")]:
    d = np.load(path)
    f1, per = weighted_f1(d["y"], d["logits"].argmax(1))
    print(f"{name:15} wF1 {f1:.3f} | " + " ".join(f"{LABELS[c]} {per[c]:.2f}" for c in range(len(LABELS))))

A01 debiased    wF1 0.594 | yor 0.88 ibo 0.99 hau 0.85 ego 0.92 anw 0.87 efi 0.11 ibb 0.00 idu 0.93 igl 0.43 nup 0.00 tiv 0.35 urh 0.97 other 0.88
A03 plain       wF1 0.607 | yor 0.88 ibo 0.98 hau 0.96 ego 0.96 anw 0.86 efi 0.12 ibb 0.00 idu 0.93 igl 0.35 nup 0.00 tiv 0.46 urh 0.99 other 0.92
soup(A01, A03)  wF1 0.626 | yor 0.90 ibo 0.98 hau 0.88 ego 0.98 anw 0.91 efi 0.18 ibb 0.03 idu 0.93 igl 0.50 nup 0.00 tiv 0.42 urh 0.98 other 0.90


## Inference

In [8]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
repo, rev = "datapsalm/dsn2026-lid-f07", "1ac06ea69be473693a29981b12cab0ec787dbba2"
model = AutoModelForAudioClassification.from_pretrained(repo, revision=rev, dtype=torch.float32).eval()
fe = AutoFeatureExtractor.from_pretrained(repo, revision=rev)

rng = np.random.default_rng(1)
times = []
with torch.no_grad():
    for c in range(len(LABELS)):
        idx = np.flatnonzero(va.labels == c)
        if not len(idx):
            continue
        i = int(rng.choice(idx))
        x = va.wav(i, 0, min(va.lengths[i], 12 * SR))
        t = time.time()
        p = model(**fe([x], sampling_rate=SR, return_tensors="pt")).logits.softmax(-1)[0]
        times.append(time.time() - t)
        print(f"{LABELS[c]:5} -> {model.config.id2label[int(p.argmax())]:5} {p.max():.2f}")
print(f"{1000*np.median(times):.0f} ms/clip (CPU)")

yor   -> yor   0.90
ibo   -> ibo   0.90
hau   -> hau   0.93


ego   -> ego   0.86
anw   -> anw   0.90
efi   -> efi   0.86


ibb   -> ibb   0.84
idu   -> idu   0.89
igl   -> igl   0.96


tiv   -> tiv   0.96
urh   -> urh   0.89
other -> other 0.91
92 ms/clip (CPU)
